# Análises e geração dos gráficos

**Tech Challenge Fase 3** · Pós-Tech em Data Analytics, FIAP
**Base:** State of Data Brazil, edições 2023-2024, 2024-2025 e 2025-2026
**Etapa do pipeline:** visualização e leitura interpretativa

Este notebook consome a camada Gold e produz os gráficos usados no material executivo. Todo gráfico traz título afirmativo, rótulos de eixo com unidade, rótulo de valor visível e fonte dos dados.

## 1. Sistema visual

A paleta é uma rampa monocromática azul de três passos, uma por edição da pesquisa. A edição é variável ordinal, portanto a rampa sequencial é a codificação correta: o tom mais claro é a edição mais antiga e o mais escuro a mais recente.

A rampa foi validada por medição, e não a olho:

In [1]:
import sys
sys.path.insert(0, "../src")
import estilo as E

def luminancia(hexa):
    r, g, b = [int(hexa[i:i+2], 16) / 255 for i in (1, 3, 5)]
    f = lambda c: c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4
    return 0.2126 * f(r) + 0.7152 * f(g) + 0.0722 * f(b)

def cinza(hexa):
    r, g, b = [int(hexa[i:i+2], 16) for i in (1, 3, 5)]
    return round(0.299 * r + 0.587 * g + 0.114 * b)

rampa = [E.AZUL_CLARO, E.AZUL_MEDIO, E.AZUL_ESCURO]
print("cor       luminancia   nivel de cinza")
for cor in rampa:
    print(f"{cor}   {luminancia(cor):.4f}      {cinza(cor)}")

lums = [luminancia(c) for c in rampa]
print("\nluminancia estritamente decrescente:", all(lums[i] > lums[i+1] for i in range(len(lums)-1)))
print("distancia minima em escala de cinza:", min(abs(cinza(rampa[i]) - cinza(rampa[i+1])) for i in range(len(rampa)-1)))

cor       luminancia   nivel de cinza
#8FBCD4   0.4656      177
#4E8098   0.1933      116
#1B3A5C   0.0403      53

luminancia estritamente decrescente: True
distancia minima em escala de cinza: 61


**Leitura do resultado.** A luminância cai de forma estritamente decrescente e a menor distância entre passos vizinhos é de 61 níveis de cinza, bem acima do mínimo de 25 adotado no projeto. Os gráficos continuam legíveis em impressão em preto e branco.

## 2. Leitura da camada Gold

In [2]:
import pandas as pd

G = "../dados/gold_csv"
# keep_default_na=False: ha categorias cujo texto literal e "NA" e "N/A", digitadas
# no campo livre de ferramenta preferida. Sem isso elas virariam valor ausente.
ler = lambda arq: pd.read_csv(f"{G}/{arq}", keep_default_na=False)
distribuicoes = ler("gold_distribuicoes.csv")
salario = ler("gold_salario.csv")
mencoes = ler("gold_mencoes.csv")
cruzamentos = ler("gold_cruzamentos.csv")

print(f"distribuicoes: {len(distribuicoes)} linhas, {distribuicoes.dimensao.nunique()} dimensoes")
print(f"cruzamentos:   {len(cruzamentos)} linhas")
print(f"salario:       {len(salario)} linhas")
print(f"mencoes:       {len(mencoes)} linhas")

distribuicoes: 1008 linhas, 25 dimensoes
cruzamentos:   982 linhas
salario:       135 linhas
mencoes:       29 linhas


## 3. Verificação estatística antes de afirmar

Um título de gráfico afirma uma conclusão, portanto precisa estar sustentado pelo dado. O teste abaixo evita afirmar liderança onde há empate técnico.

In [3]:
import math

setor = distribuicoes[(distribuicoes.dimensao == "setor") & (distribuicoes.edicao == "2025-2026")].nlargest(2, "participacao_pct")
n = int(setor.total_validos.iloc[0])
p1, p2 = setor.participacao_pct.iloc[0] / 100, setor.participacao_pct.iloc[1] / 100
erro_padrao = math.sqrt(p1 * (1 - p1) / n + p2 * (1 - p2) / n) * 100
diferenca = (p1 - p2) * 100

print(f"1o lugar: {setor.categoria.iloc[0]} com {setor.participacao_pct.iloc[0]:.1f}%")
print(f"2o lugar: {setor.categoria.iloc[1]} com {setor.participacao_pct.iloc[1]:.1f}%")
print(f"diferenca: {diferenca:.1f} pontos | erro padrao: {erro_padrao:.2f} | z = {diferenca/erro_padrao:.2f}")
print("conclusao:", "diferenca significativa" if abs(diferenca / erro_padrao) > 1.96 else "empate tecnico a 95% de confianca")

1o lugar: Finanças ou Bancos com 18.5%
2o lugar: Tecnologia/Fábrica de Software com 17.4%
diferenca: 1.1 pontos | erro padrao: 0.96 | z = 1.13
conclusao: empate tecnico a 95% de confianca


**Leitura do resultado.** A diferença entre o primeiro e o segundo setor não é significativa a 95% de confiança. Por isso o gráfico correspondente afirma que os dois somam mais de um terço do mercado, e não que um lidera o outro.

O mesmo cuidado se aplica ao gráfico de satisfação por modelo de trabalho, em que dois modelos aparecem muito próximos no topo.


In [4]:
sat = cruzamentos[(cruzamentos.dimensao_1 == "modelo_trabalho")
                  & (cruzamentos.dimensao_2 == "satisfacao")
                  & (cruzamentos.edicao == "2025-2026")
                  & (cruzamentos.categoria_2 == "Sim")].nlargest(2, "participacao_pct")

x1, n1 = int(sat.respondentes.iloc[0]), int(sat.total_no_grupo.iloc[0])
x2, n2 = int(sat.respondentes.iloc[1]), int(sat.total_no_grupo.iloc[1])
p_comum = (x1 + x2) / (n1 + n2)
z = (x1 / n1 - x2 / n2) / math.sqrt(p_comum * (1 - p_comum) * (1 / n1 + 1 / n2))

for i in range(2):
    rotulo = sat.categoria_1.iloc[i].split(" (")[0]
    print(f"{rotulo:42} {sat.participacao_pct.iloc[i]:5.2f}%  (n={int(sat.total_no_grupo.iloc[i])})")
print(f"\ndiferenca: {sat.participacao_pct.iloc[0] - sat.participacao_pct.iloc[1]:.2f} ponto")
print(f"z: {abs(z):.2f}   significativa a 95%: {'sim' if abs(z) > 1.96 else 'nao'}")


Modelo híbrido flexível                    74.96%  (n=631)
Modelo 100% remoto                         74.47%  (n=1281)

diferenca: 0.49 ponto
z: 0.23   significativa a 95%: nao


**Leitura do resultado.** O híbrido flexível e o remoto integral empatam: a diferença de 0,5 ponto tem z igual a 0,23, muito abaixo do 1,96 exigido para 95% de confiança. Por isso o título do gráfico afirma que a flexibilidade, e não o remoto puro, separa satisfeitos de insatisfeitos, e não que o remoto lidera. A separação real está entre ter e não ter flexibilidade, com 20,9 pontos entre o híbrido flexível e o presencial integral.


## 4. Geração dos gráficos

In [5]:
import subprocess

resultado = subprocess.run([sys.executable, "../src/graficos.py"], capture_output=True, text=True, cwd="..")
print(resultado.stdout[-700:])

## 5. Diagrama da arquitetura

In [6]:
resultado = subprocess.run([sys.executable, "../src/diagrama.py"], capture_output=True, text=True, cwd="..")
print(resultado.stdout.strip())

## 6. Conferência final dos arquivos gerados

In [7]:
from pathlib import Path

figuras = sorted(Path("../figuras").glob("*.png"))
for figura in figuras:
    print(f"{figura.name:32} {figura.stat().st_size/1024:7.1f} KB")
print(f"\ntotal: {len(figuras)} imagens")

arquitetura_aws.png                218.8 KB
fig01_genero_serie.png             112.9 KB
fig02_genero_nivel.png             106.4 KB
fig03_salario_nivel.png            131.4 KB
fig04_modelo_trabalho.png          118.0 KB
fig05_ia_prioridade.png            139.9 KB
fig06_ia_serie.png                 145.3 KB
fig07_linguagens.png               120.5 KB
fig08_setor.png                    143.8 KB
fig09_regiao.png                   123.0 KB
fig10_satisfacao_modelo.png        134.5 KB
fig11_ia_impacto.png               122.5 KB
fig12_salario_cargo.png            197.0 KB
fig13_ferramentas.png              126.5 KB

total: 14 imagens
